<a href="https://colab.research.google.com/github/bru-or/mydailystudies/blob/main/Converter%20bibtex%20para%20csv%20e%20xlsx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conversor

In [ ]:
!pip install bibtexparser
!pip install xlsxwriter # Ensure xlsxwriter is installed for Excel output

import bibtexparser
import pandas as pd
from google.colab import files
import io

print("Por favor, faça o upload do seu arquivo BibTeX (.bib):")
uploaded = files.upload()

if not uploaded:
    print("Nenhum arquivo foi carregado.")
else:
    for filename, content in uploaded.items():
        print(f"Arquivo '{filename}' carregado com sucesso.")
        try:
            # Decode the content to string (assuming UTF-8)
            bibtex_str = content.decode('utf-8')

            # Load the BibTeX database
            bib_database = bibtexparser.loads(bibtex_str)

            # Convert to Pandas DataFrame
            df = pd.DataFrame(bib_database.entries)

            # Rename 'ID' column to 'cod referencia'
            if 'ID' in df.columns:
                df = df.rename(columns={'ID': 'cod referencia'})
            else:
                df['cod referencia'] = '' # Add an empty column if ID is not found

            # Define the desired column order and ensure all exist, adding empty if not
            desired_columns = [
                "cod referencia", "author", "title", "year", "isbn", "publisher",
                "address", "url", "doi", "abstract", "booktitle", "articleno",
                "pages", "numpages", "keywords", "location", "series"
            ]

            # Add missing columns with empty string values to the DataFrame
            for col in desired_columns:
                if col not in df.columns:
                    df[col] = ''

            # Select and reorder columns
            df = df[desired_columns]

            # Fill any remaining NaN values with empty strings
            df = df.fillna('')

            # Prepare CSV for download
            csv_filename = filename.replace('.bib', '.csv')
            csv_output = df.to_csv(index=False, encoding='utf-8')
            with open(csv_filename, 'w', encoding='utf-8') as f:
                f.write(csv_output)
            files.download(csv_filename)
            print(f"Arquivo '{csv_filename}' gerado e pronto para download.")

            # Prepare XLSX for download
            xlsx_filename = filename.replace('.bib', '.xlsx')
            with pd.ExcelWriter(xlsx_filename, engine='xlsxwriter') as writer:
                df.to_excel(writer, index=False, sheet_name='BibTeX Data')
            files.download(xlsx_filename)
            print(f"Arquivo '{xlsx_filename}' gerado e pronto para download.")

        except Exception as e:
            print(f"Erro ao processar o arquivo '{filename}': {e}")


Por favor, faça o upload do seu arquivo BibTeX (.bib):
